# qSPECT Tutorial

This tutorial converts a reconstructed SPECT DICOM image from scanner counts into a quantitative SPECT (qSPECT) image in Bq/mL using `DicomModify.make_bqml_suv`. It also adds the patient and radiopharmaceutical metadata needed by software that calculates standardized uptake values (SUVs).

> **Before you begin:** the input must be a reconstructed SPECT image whose stored pixels represent counts. The method currently supports Siemens and GE DICOM data and expects acquisition timing, projection, voxel-size, and manufacturer metadata to be present. Do not use an image that the scanner has already calibrated into activity concentration.

## What the conversion does

For each voxel, `make_bqml_suv` converts scanner counts to Bq/mL using the acquisition frame duration, the number of projections, a camera calibration factor, and the voxel volume:

$$A_{voxel} [Bq/mL] = \frac{counts}{frame\ duration \times projections} \times CF \times \frac{10^6}{voxel\ volume [mL]}$$

Here, `CF` is in MBq/(counts/s). The method rescales the result into signed 16-bit DICOM pixels, records the Bq/mL real-world value mapping, adds injection and patient metadata, applies decay-correction metadata, and assigns new SOP and Series Instance UIDs. The returned table is a summary of the injection and scan timing; the modified image remains in `image.ds` until it is saved.

In [7]:
from pytheranostics.dicomtools.dicomtools import DicomModify

## 1. Select the reconstructed counts image

Enter the path to the reconstructed SPECT DICOM image exported by the scanner in counts. The output path is created alongside the input with `_qspect` appended to its filename, leaving the source image unchanged. You may set `output_path` to a different location if preferred.

In [8]:
# Replace this with the path to your reconstructed SPECT DICOM in counts.
spect_counts_path = "../../examples/data/testimages/016.dcm" #Path("/path/to/reconstructed_spect_counts.dcm")

output_path = "./016_qspect.dcm"

print(f"Input:  {spect_counts_path}")
print(f"Output: {output_path}")

Input:  ../../examples/data/testimages/016.dcm
Output: ./016_qspect.dcm


## 2. Supply the camera calibration factor

Use the site-, camera-, collimator-, energy-window-, isotope-, and reconstruction-specific calibration factor established by your quantitative imaging protocol. Its required unit is **MBq/(counts/s)**. The value below belongs only to the example dataset and must not be reused for clinical data without validation.

In [9]:
calibration_factor = 0.10800584442987242  # MBq/(counts/s); example only
image = DicomModify(str(spect_counts_path), CF=calibration_factor)

## 3. Enter patient and injection information

Transcribe these values from the administration record. Activities are in MBq, weight is in kg, and **height is in cm**. Dates use `YYYYMMDD`; times use 24-hour `HHMM`.

The method decay-corrects both syringe measurements to the injection time and subtracts the residual activity to estimate the net injected activity. The default half-life is 574,300 seconds (Lu-177), and the default radiopharmaceutical label is `Lutetium-PSMA-617`. Pass different values when processing another radionuclide or agent.

In [10]:
weight_kg = 113.4
height_cm = 178.5

injection_date = "20220616"
pre_inj_activity_mbq = 7450.0
pre_inj_time = "0804"
post_inj_activity_mbq = 14.4
post_inj_time = "0955"
injection_time = "0918"

# Use a validated correction here only if the dose calibrator requires one.
activity_meter_scale_factor = 1.0

## 4. Convert the image and inspect the summary

This call modifies the in-memory DICOM dataset. For Siemens data, `n_detectors` defaults to 2 and is used with `NumberOfFramesInRotation` to recover the total projection count. Set it explicitly if the acquisition used a different detector configuration.

In [11]:
injection_summary = image.make_bqml_suv(
    weight=weight_kg,
    height=height_cm,
    injection_date=injection_date,
    pre_inj_activity=pre_inj_activity_mbq,
    pre_inj_time=pre_inj_time,
    post_inj_activity=post_inj_activity_mbq,
    post_inj_time=post_inj_time,
    injection_time=injection_time,
    activity_meter_scale_factor=activity_meter_scale_factor,
)

injection_summary

/home/esquinas/projects/venvs/.dosimetry/lib/python3.14/site-packages/pydicom/valuerep.py:440: UserWarning: The value length (17) exceeds the maximum length of 16 allowed for VR DS. Please see <https://dicom.nema.org/medical/dicom/current/output/html/part05.html#table_6.2-1> for allowed values for each VR.
  warn_and_log(msg)


,patient_id,weight_kg,height_cm,pre_inj_activity_MBq,pre_inj_datetime,post_inj_activity_MBq,post_inj_datetime,injected_activity_MBq,injection_datetime,scan_datetime,delta_t_days
0,PR21-CAVA-0016,113.4,178.5,7450.0,2022-06-16 08:04:00,14.4,2022-06-16 09:55:00,7395.744895,2022-06-16 09:18:00,2022-06-17 11:37:04.478,1.09658


## 5. Save and verify the qSPECT DICOM

Save to the explicit output path only after the conversion succeeds. Then reopen the file and verify the series description and real-world mapping. Quantitative use should also include your site's usual image- and metadata-QC checks.

In [12]:
image.ds.save_as(output_path)

mapping = image.ds.RealWorldValueMappingSequence[0]
print(f"Saved: {output_path}")
print(f"Series description: {image.ds.SeriesDescription}")
print(f"Mapped units: {mapping.MeasurementUnitsCodeSequence[0].CodeValue}")
print(f"Real-world slope: {mapping.RealWorldValueSlope}")
print(f"Real-world intercept: {mapping.RealWorldValueIntercept}")

Saved: ./016_qspect.dcm
Series description: QSPECT_Lu PSMA 617,6/17/2022 [WB Recon - AC ]
Mapped units: Bq/ml
Real-world slope: 51.68902587890625
Real-world intercept: 0.0


## Quality-control checklist

Before using the result for quantification, confirm that:

- the input pixels were counts and the reconstruction matches the calibration-factor protocol;
- the DICOM manufacturer, frame duration, number of projections, and voxel dimensions are correct;
- patient weight and height use kg and cm, respectively;
- syringe activities, measurement times, injection time, half-life, and radiopharmaceutical are correct;
- the calculated injected activity and scan-to-injection interval in `injection_summary` are plausible; and
- a known phantom or other site-approved test confirms the output values in Bq/mL.

The DICOM stores integer pixels. Consumers must apply the `RealWorldValueSlope` and `RealWorldValueIntercept` in `RealWorldValueMappingSequence` to recover Bq/mL values.